In [40]:
!pip install fastapi uvicorn pyngrok nest_asyncio sentence-transformers

In [41]:
import pandas as pd
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [42]:
menu = [
    {"dish": "Pasta Alfredo", "desc": "creamy cheesy pasta comfort food", "category": "main"},
    {"dish": "Margherita Pizza", "desc": "cheesy pizza with tomato base", "category": "main"},
    {"dish": "Chicken Biryani", "desc": "spicy rice with chicken indian masala", "category": "main"},
    {"dish": "Veg Salad", "desc": "healthy light vegetarian salad", "category": "main"},
    {"dish": "Chocolate Lava Cake", "desc": "warm chocolate dessert comfort sweet", "category": "dessert"},
    {"dish": "Ice Cream Sundae", "desc": "cold sweet dessert creamy", "category": "dessert"},
    {"dish": "Cold Coffee", "desc": "chilled coffee beverage sweet", "category": "drink"},
    {"dish": "Mango Smoothie", "desc": "fresh fruity drink sweet", "category": "drink"}
]

df = pd.DataFrame(menu)

In [43]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['desc'].tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
def detect_intent(user_input):
    text = user_input.lower()

    if "sad" in text or "bad day" in text:
        return "comfort"
    elif "healthy" in text or "diet" in text:
        return "healthy"
    elif "spicy" in text:
        return "spicy"
    elif "late night" in text:
        return "late_night"
    else:
        return "general"

In [45]:
def upsell(dish):
    d = dish.lower()

    if "biryani" in d:
        return {"add_on": "Gulab Jamun", "reason": "perfect dessert after spicy meal"}
    elif "pizza" in d:
        return {"add_on": "Garlic Bread", "reason": "best combo with pizza"}
    elif "dessert" in d or "cake" in d:
        return {"add_on": "Ice Cream", "reason": "enhances dessert experience"}
    else:
        return {"add_on": "Cold Drink", "reason": "completes your meal"}

In [46]:
app = FastAPI()

class UserRequest(BaseModel):
    message: str

def recommend(user_input):
    try:
        user_emb = model.encode([user_input])
        scores = cosine_similarity(user_emb, embeddings)[0]

        top_idx = scores.argsort()[-3:][::-1]

        results = []
        for i in top_idx:
            results.append({
                "dish_name": df.iloc[i]['dish'],
                "category": df.iloc[i]['category'],
                "confidence": float(scores[i])
            })

        return results

    except Exception as e:
        print("Recommendation error:", e)
        return []

@app.post("/chat")
async def chat(req: UserRequest):
    try:
        user_input = req.message
        print("👉 User Input:", user_input)

        # Check model
        if model is None:
            return {"error": "Model not loaded"}

        # Check embeddings
        if embeddings is None:
            return {"error": "Embeddings not created"}

        # Detect intent
        intent = detect_intent(user_input)
        print(" Intent:", intent)

        # Recommend
        recs = recommend(user_input)
        print(" Recs:", recs)

        if not recs or len(recs) == 0:
            return {
                "mcp_version": "1.0",
                "intent": intent,
                "recommendations": [],
                "upsell": None
            }

        # Upsell
        upsell_data = upsell(recs[0]["dish_name"])

        return {
            "mcp_version": "1.0",
            "intent": intent,
            "user_input": user_input,
            "recommendations": recs,
            "upsell": upsell_data
        }

    except Exception as e:
        import traceback
        traceback.print_exc()   #error print
        return {
            "error": str(e)
        }
    return response

In [47]:
import nest_asyncio
import uvicorn
import threading
from google.colab.output import eval_js

nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start server in background
thread = threading.Thread(target=run)
thread.start()

# Get public Colab URL
public_url = eval_js("google.colab.kernel.proxyPort(8000)")
print(" API Live at:", public_url)

INFO:     Started server process [1241]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🚀 API Live at: https://8000-gpu-t4-s-kkb-usw4b0-f7bl54snyrbw-b.us-west4-0.prod.colab.dev
